<a href="https://colab.research.google.com/github/OmerSalihS/BEV_Parking_Proj/blob/main/BEV_Parking_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
!pip install mmdet mmdet3d
!pip install opencv-python pillow matplotlib numpy pandas opencv-contrib-python
!pip install onnx onnxruntime-gpu  # For ONNX conversion later

print("✓ Dependencies installed!")

Looking in indexes: https://download.pytorch.org/whl/cu118
Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.9/607.9 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 26.8 MB/s eta 0:00:00


In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import Image, display

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️  No GPU detected. Please enable GPU in Runtime settings.")

In [ ]:
def detect_vehicles_simple(image_path):
    """
    Simple vehicle detection using OpenCV.
    This is a starting point - you'll upgrade to BEV model later.
    """
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Simple detection using HOG (Histogram of Oriented Gradients)
    hog = cv2.HOGDescriptor()
    hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

    # Detect objects
    boxes, weights = hog.detectMultiScale(img_gray, winStride=(8, 8), padding=(32, 32), scale=1.05)

    # Draw detections
    result_img = img_rgb.copy()
    detections = []

    for (x, y, w, h) in boxes:
        cv2.rectangle(result_img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        detections.append({
            'bbox': [x, y, w, h],
            'confidence': 0.8
        })

    return result_img, detections

print("✓ Detection function ready")

In [ ]:
def create_parking_grid(image_shape, grid_size=(10, 5)):
    """
    Create a grid of parking spaces.
    """
    h, w = image_shape[:2]
    rows, cols = grid_size

    spaces = []
    space_h = h // rows
    space_w = w // cols

    for i in range(rows):
        for j in range(cols):
            x1 = j * space_w
            y1 = i * space_h
            x2 = (j + 1) * space_w
            y2 = (i + 1) * space_h

            spaces.append({
                'id': f"{i}_{j}",
                'bbox': [x1, y1, x2, y2],
                'occupied': False
            })

    return spaces

def check_parking_occupancy(spaces, vehicle_detections):
    """
    Check which parking spaces are occupied.
    """
    for space in spaces:
        space_bbox = space['bbox']
        space_center = [(space_bbox[0] + space_bbox[2]) / 2,
                       (space_bbox[1] + space_bbox[3]) / 2]

        for vehicle in vehicle_detections:
            vehicle_bbox = vehicle['bbox']
            if (space_center[0] >= vehicle_bbox[0] and
                space_center[0] <= vehicle_bbox[0] + vehicle_bbox[2] and
                space_center[1] >= vehicle_bbox[1] and
                space_center[1] <= vehicle_bbox[1] + vehicle_bbox[3]):
                space['occupied'] = True
                break

    return spaces

print("✓ Parking grid functions ready")

In [ ]:
def visualize_parking_lot(image, spaces, detections):
    """
    Visualize parking lot with grid and occupancy status.
    """
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    ax.imshow(image)

    # Draw parking grid
    for space in spaces:
        x1, y1, x2, y2 = space['bbox']
        color = 'red' if space['occupied'] else 'green'
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                           fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)

        # Add space ID
        ax.text(x1 + 10, y1 + 20, space['id'],
               color='white', fontsize=8, weight='bold',
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))

    # Draw vehicle detections
    for det in detections:
        x, y, w, h = det['bbox']
        rect = plt.Rectangle((x, y), w, h,
                           fill=False, edgecolor='blue', linewidth=2)
        ax.add_patch(rect)

    occupied_count = sum(s['occupied'] for s in spaces)
    ax.set_title(f"Parking Lot Analysis - {occupied_count}/{len(spaces)} occupied")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ Visualization function ready")

In [ ]:
# 1. Upload a parking lot image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# 2. Display uploaded image
print("Uploaded image:")
display(Image(image_path))

# 3. Load and detect vehicles
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
result_img, detections = detect_vehicles_simple(image_path)

# 4. Create parking grid
spaces = create_parking_grid(img_rgb.shape, grid_size=(5, 10))

# 5. Check occupancy
spaces = check_parking_occupancy(spaces, detections)

# 6. Visualize results
visualize_parking_lot(img_rgb, spaces, detections)

# 7. Print statistics
total = len(spaces)
occupied = sum(s['occupied'] for s in spaces)
available = total - occupied
utilization = (occupied / total) * 100

print(f"\n📊 Parking Statistics:")
print(f"Total spaces: {total}")
print(f"Occupied: {occupied}")
print(f"Available: {available}")
print(f"Utilization: {utilization:.1f}%")